**Purpose**
---

Load normalized datasets from 01_prepare_datasets.ipynb, generate chunks using multiple chunking strategies, and persist them for embedding generation.

**Chunking Strategies**

We will benchmark three chunking methods:

CHUNKING_METHODS = [
    "standard",
    "metadata",
    "small2big"
]

In [5]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Read datasets from existing location
DATASET_DIR = "/content/content/datasets"

# Save generated chunks to Drive
PROJECT_ROOT = "/content/drive/MyDrive/RAGBenchmark"


DATASET_DIR = "/content/content/datasets"

CHUNK_OUTPUT_DIR = "/content/drive/MyDrive/RAGBenchmark/chunks"


CHUNK_OUTPUT_DIR = os.path.join(PROJECT_ROOT, "chunks")

os.makedirs(CHUNK_OUTPUT_DIR, exist_ok=True)

print("Reading datasets from :", DATASET_DIR)
print("Saving chunks to      :", CHUNK_OUTPUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Reading datasets from : /content/content/datasets
Saving chunks to      : /content/drive/MyDrive/RAGBenchmark/chunks


In [6]:
chunk_count = 0

for root, dirs, files in os.walk(CHUNK_OUTPUT_DIR):
    for f in files:
        if f.endswith(".pkl"):
            chunk_count += 1

print(chunk_count)

0


In [7]:
#Install dependencies

!pip install -q pandas pyarrow tqdm nltk

In [8]:
#Imports

import os
import json
import pickle
import pandas as pd
from tqdm.auto import tqdm
from nltk.tokenize import sent_tokenize

import nltk
nltk.download("punkt")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [9]:
import os

PROJECT_ROOT = "/content/drive/MyDrive/RAGBenchmark"

DATASET_DIR = os.path.join(PROJECT_ROOT, "datasets")
CHUNK_DIR = os.path.join(PROJECT_ROOT, "chunks")
EMBEDDING_DIR = os.path.join(PROJECT_ROOT, "embeddings")
INDEX_DIR = os.path.join(PROJECT_ROOT, "indexes")
EXPERIMENT_DIR = os.path.join(PROJECT_ROOT, "experiments")
JUDGE_CACHE_DIR = os.path.join(PROJECT_ROOT, "judge_cache")

for folder in [
    DATASET_DIR,
    CHUNK_DIR,
    EMBEDDING_DIR,
    INDEX_DIR,
    EXPERIMENT_DIR,
    JUDGE_CACHE_DIR,
]:
    os.makedirs(folder, exist_ok=True)

print(PROJECT_ROOT)

/content/drive/MyDrive/RAGBenchmark


In [10]:
#Configuration



os.makedirs(CHUNK_OUTPUT_DIR, exist_ok=True)

CHUNK_SIZE = 512
SMALL_CHUNK_SIZE = 128
OVERLAP = 50

In [11]:
#Helper functions
def estimate_tokens(text):
  return max(1, len(text.split()))

In [12]:
#Sliding Window chunking

def sliding_chunks (text, chunk_size = 512, overlap=50):
  words = text.split()
  chunks=[]
  start = 0
  while start<len(words):
    end = start + chunk_size

    chunk = "".join(words[start:end])

    chunks.append(chunk)

    start += (chunk_size - overlap)

  return chunks

In [13]:
# Standard Chunking
#Fixed-size overlapping chunks


def standard_chunking(context):
  if context is None:
    return []

  if isinstance(context,list):
    context = "".join(map(str,context))

  return sliding_chunks(
      str(context),
      chunk_size=CHUNK_SIZE,
      overlap = OVERLAP
  )

In [14]:
#Metadata chunking
#Attach question and dataset metadata


def metadata_chunking(row):

  context = row["context"]

  if context is None:
    return []

  if isinstance(context,list):
    context = "".join(map(str,context))

  base_chunks = sliding_chunks(
      str(context),
      chunk_size = CHUNK_SIZE,
      overlap = OVERLAP
  )

  enriched = []

  prefix = (
      f"Dataset: {row['dataset']}\n"
      f"Question: {row['question']}\n\n"
  )

  for chunk in base_chunks:
    enriched.append(prefix + chunk)

  return enriched


In [15]:
#Small2Big Chunking

def small2big_chunking(context):
  if context is None:
    return []
  if isinstance(context, list):
    context = " ".join(map(str, context))

  small_chunks = sliding_chunks(
       str(context),
       chunk_size=SMALL_CHUNK_SIZE,
       overlap=25
       )
  grouped = []
  current = []
  for chunk in small_chunks:
   current.append(chunk)
   if len(current) == 4:
    grouped.append("\n".join(current))
    current = []
  if current:
    grouped.append("\n".join(current))

  return grouped

In [17]:
#Generate Chunks

summary = []
for domain in os.listdir(DATASET_DIR):
  domain_path = os.path.join(DATASET_DIR, domain)

  if not os.path.isdir(domain_path):
    continue

  output_domain = os.path.join(
      CHUNK_OUTPUT_DIR,
      domain
  )

  os.makedirs(output_domain, exist_ok=True)

  for file in os.listdir(domain_path):
    if not file.endswith(".parquet"):
      continue

    dataset_path = os.path.join(domain_path,file)
    dataset_name = file.replace("_test.parquet", "")

    print("=" * 60)
    print(dataset_name)

    df = pd.read_parquet(dataset_path)

    chunk_store = {
        "standard": [],
        "metadata": [],
        "small2big": []
        }

    for _, row in tqdm(
        df.iterrows(),
        total=len(df),
        desc=dataset_name
    ):

     row_dict = row.to_dict()

    chunk_store["standard"].append({
          "id": row["id"],
          "chunks": standard_chunking(row["context"])
      })

    chunk_store["metadata"].append({
          "id": row["id"],
          "chunks": metadata_chunking(row_dict)
      })

    chunk_store["small2big"].append({
          "id": row["id"],
          "chunks": small2big_chunking(row["context"])
      })




    for method in chunk_store:

      save_path = os.path.join(
          output_domain,
          f"{dataset_name}_{method}.pkl"
      )


      with open(save_path, "wb") as f:
        pickle.dump(
            chunk_store[method],
            f
        )

      total_chunks = sum(
          len(x)
          for x in chunk_store[method]
      )


      summary.append({
          "domain": domain,
          "dataset": dataset_name,
          "method": method,
          "records": len(df),
          "chunks": total_chunks,
          "path": save_path
      })

      print(
          method,
          "saved:",
          total_chunks,
          "chunks"
      )



covidqa


covidqa:   0%|          | 0/246 [00:00<?, ?it/s]

standard saved: 2 chunks
metadata saved: 2 chunks
small2big saved: 2 chunks
pubmedqa


pubmedqa:   0%|          | 0/2450 [00:00<?, ?it/s]

standard saved: 2 chunks
metadata saved: 2 chunks
small2big saved: 2 chunks
hotpotqa


...

standard saved: 2 chunks
metadata saved: 2 chunks
small2big saved: 2 chunks
finqa


finqa:   0%|          | 0/2294 [00:00<?, ?it/s]

standard saved: 2 chunks
metadata saved: 2 chunks
small2big saved: 2 chunks
tatqa


tatqa:   0%|          | 0/3338 [00:00<?, ?it/s]

standard saved: 2 chunks
metadata saved: 2 chunks
small2big saved: 2 chunks


In [18]:
for root, dirs, files in os.walk(CHUNK_OUTPUT_DIR):
    print(root)
    for f in files:
        print("   ", f)

/content/drive/MyDrive/RAGBenchmark/chunks
/content/drive/MyDrive/RAGBenchmark/chunks/biomedical
    covidqa_standard.pkl
    covidqa_metadata.pkl
    covidqa_small2big.pkl
    pubmedqa_standard.pkl
    pubmedqa_metadata.pkl
    pubmedqa_small2big.pkl
/content/drive/MyDrive/RAGBenchmark/chunks/general
    hotpotqa_standard.pkl
    hotpotqa_metadata.pkl
    hotpotqa_small2big.pkl
    msmarco_standard.pkl
    msmarco_metadata.pkl
    msmarco_small2big.pkl
    hagrid_standard.pkl
    hagrid_metadata.pkl
    hagrid_small2big.pkl
    expertqa_standard.pkl
    expertqa_metadata.pkl
    expertqa_small2big.pkl
/content/drive/MyDrive/RAGBenchmark/chunks/legal
    cuad_standard.pkl
    cuad_metadata.pkl
    cuad_small2big.pkl
/content/drive/MyDrive/RAGBenchmark/chunks/support
    delucionqa_standard.pkl
    delucionqa_metadata.pkl
    delucionqa_small2big.pkl
    emanual_standard.pkl
    emanual_metadata.pkl
    emanual_small2big.pkl
    techqa_standard.pkl
    techqa_metadata.pkl
    techqa_sma

In [19]:
import os

chunk_count = 0

for root, dirs, files in os.walk("/content/drive/MyDrive/RAGBenchmark/chunks"):
    for f in files:
        if f.endswith(".pkl"):
            chunk_count += 1

print("Chunk files:", chunk_count)

Chunk files: 36


In [21]:
summary_df = pd.DataFrame(summary)

summary_df.to_csv(
    "/content/drive/MyDrive/RAGBenchmark/chunks/chunk_summary.csv",
    index=False
)

print(summary_df.head())
print("Chunks persisted to Drive.")

       domain   dataset     method  records  chunks  \
0  biomedical   covidqa   standard      246       2   
1  biomedical   covidqa   metadata      246       2   
2  biomedical   covidqa  small2big      246       2   
3  biomedical  pubmedqa   standard     2450       2   
4  biomedical  pubmedqa   metadata     2450       2   

                                                path  
0  /content/drive/MyDrive/RAGBenchmark/chunks/bio...  
1  /content/drive/MyDrive/RAGBenchmark/chunks/bio...  
2  /content/drive/MyDrive/RAGBenchmark/chunks/bio...  
3  /content/drive/MyDrive/RAGBenchmark/chunks/bio...  
4  /content/drive/MyDrive/RAGBenchmark/chunks/bio...  
Chunks persisted to Drive.
